# Preprocesamiento del dataset

In [1]:
# Instalamos Roboflow para poder descargar el dataset desde Roboflow Universe
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 77.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [2]:
# Importamos las librerías necesarias
import os
import shutil
import yaml

from roboflow import Roboflow
from google.colab import userdata

In [3]:
# Descargamos el dataset original desde Roboflow en formato YOLOv8
api_key = userdata.get("RoboflowAPIKey")

rf = Roboflow(api_key=api_key)

project = rf.workspace("raiyan8018").project("sunflower-mn2cr")

dataset = project.version(1).download("yolov8")

original_dataset_path = dataset.location

print("Dataset original descargado en:")
print(original_dataset_path)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Sunflower-1 in yolov8:: 100%|██████████| 8584/8584 [00:03<00:00, 2416.58it/s]

Dataset original descargado en:
/content/Sunflower-1


In [4]:
# Verificamos la estructura del dataset original
os.listdir(original_dataset_path)

['test',
 'valid',
 'train',
 'data.yaml',
 'README.roboflow.txt',
 'README.dataset.txt']

In [5]:
# Consultamos cuánto ocupa el dataset original antes de clonarlo
!du -sh {original_dataset_path}

360M	/content/Sunflower-1


### Clonación del dataset

In [6]:
# Creamos una copia del dataset original para no modificar los datos descargados desde Roboflow
unified_dataset_path = "/content/sunflower_unified"

if os.path.exists(unified_dataset_path):
    shutil.rmtree(unified_dataset_path)

shutil.copytree(original_dataset_path, unified_dataset_path)

print("Dataset clonado correctamente en:")
print(unified_dataset_path)

Dataset clonado correctamente en:
/content/sunflower_unified


In [7]:
# Verificamos que la copia tenga la misma estructura que el dataset original
os.listdir(unified_dataset_path)

['test',
 'valid',
 'train',
 'data.yaml',
 'README.roboflow.txt',
 'README.dataset.txt']

### Unificación de clases

In [8]:
# Unificamos las clases del dataset: cualquier class_id pasa a ser 0
for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)

        new_lines = []

        with open(label_path, "r") as file:
            lines = file.readlines()

        for line in lines:
            parts = line.strip().split()

            if len(parts) == 5:
                parts[0] = "0"
                new_lines.append(" ".join(parts))

        with open(label_path, "w") as file:
            file.write("\n".join(new_lines))

print("Clases unificadas correctamente.")

Clases unificadas correctamente.


In [9]:
# Actualizamos el archivo data.yaml para indicar que ahora existe una sola clase
yaml_path = os.path.join(unified_dataset_path, "data.yaml")

with open(yaml_path, "r") as file:
    data_yaml = yaml.safe_load(file)

data_yaml["nc"] = 1
data_yaml["names"] = ["sunflower"]

with open(yaml_path, "w") as file:
    yaml.dump(data_yaml, file, sort_keys=False)

print("data.yaml actualizado correctamente.")

data.yaml actualizado correctamente.


In [10]:
# Mostramos el contenido actualizado de data.yaml
with open(yaml_path, "r") as file:
    print(file.read())

names:
- sunflower
nc: 1
roboflow:
  license: CC BY 4.0
  project: sunflower-mn2cr
  url: https://universe.roboflow.com/raiyan8018/sunflower-mn2cr/dataset/1
  version: 1
  workspace: raiyan8018
test: ../test/images
train: ../train/images
val: ../valid/images



In [11]:
# Verificamos que todos los labels tengan únicamente class_id = 0
class_ids = set()

for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)

        with open(label_path, "r") as file:
            for line in file:
                parts = line.strip().split()

                if len(parts) == 5:
                    class_id = int(parts[0])
                    class_ids.add(class_id)

print("Clases presentes después de unificar:")
print(class_ids)

Clases presentes después de unificar:
{0}


In [12]:
# Verificamos nuevamente cantidad de imágenes y labels en cada partición del dataset unificado
for split in ["train", "valid", "test"]:
    images_dir = os.path.join(unified_dataset_path, split, "images")
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    n_images = len(os.listdir(images_dir))
    n_labels = len(os.listdir(labels_dir))

    print(f"{split}: {n_images} imágenes | {n_labels} labels")

train: 3429 imágenes | 3429 labels
valid: 428 imágenes | 428 labels
test: 429 imágenes | 429 labels


# Entrenamiento